# MEG Faces Spectral Envelopes

This is a companion tutorial to the paper *"A Primer on Low-Dimensional Neural Dynamics: PCA-Based Trajectory Analysis for EEG and MEG."*

The MEG main-analysis tutorial followed the **broadband evoked field**: signed, phase-locked deflections that survive averaging. That is only part of what MEG measures. A large share of task-related activity is **induced** — it changes oscillatory amplitude at a consistent latency but with an inconsistent phase, so it cancels almost completely in an ERF average.

This notebook applies the same PCA trajectory workflow to that other half of the signal, asking how **oscillatory amplitude** evolves for **Famous faces**, **Unfamiliar faces** and **Scrambled images** in three bands:

- **Alpha:** 8–12 Hz
- **Beta:** 13–30 Hz
- **Low gamma:** 30–45 Hz

Each band receives its own shared PCA, optional participant-specific PCAs, and the same planned comparisons used in the broadband tutorial. The sequence deliberately separates **representation construction**, **geometry**, **participant-level metrics** and **inference**, so that a visually smooth trajectory is never mistaken for evidence by itself.

<div class="alert alert-secondary">
<b>🔄 What changes when you change the representation:</b><br>
A point in state space no longer means "the field pattern right now". It means "the baseline-relative band power pattern, smoothed over 40 ms, right now". Filtering, the Hilbert modulus, the squaring to power, the smoothing and the dB baseline <i>all</i> change what a coordinate is — and they all impose temporal smoothness that the raw signal did not have. These trajectories are not tidier versions of an ERF; they answer a different question.
</div>

### Why band power needs its own preprocessing

Two constraints force a separate derivative set rather than reusing the broadband epochs:

1. **Filters need room.** A narrowband FIR filter has a long impulse response; applied to a short epoch, its edge transients land inside the window you intend to interpret. The alpha filter is the longest, and Step 1 prints its actual support rather than assuming.
2. **Baselining happens in a different place.** A voltage/field baseline subtracts a mean *before* the transform. Power is baselined *after* the Hilbert step, as a ratio. Applying both would remove the very slow structure the envelope is meant to measure.

<div class="alert alert-secondary">
<b>🗺️ Position in the tutorial series:</b><br>
The EEGBCI notebook introduces the core 10-step workflow, the MEG main analysis repeats it on whitened 306-sensor data, and this notebook repeats it once more on a spectral representation of the same trials. The MEG decoding notebook then asks whether any of it predicts single trials in a new participant.
</div>

## 0. Setup & Configuration

Every scientific choice is declared before any data is touched. Because band power requires longer epochs than the broadband derivatives provide, this notebook can also *create* its own derivative set on first run.

<div class="alert alert-info">
<b>⚙️ The visible analysis contract:</b><br>
The sensor set, bands, crop, baseline, envelope sampling rate, smoothing width, active window, participant list, PCA mode and random seed are all declared in the next cell. Change them there rather than inside later analysis cells — a parameter edited mid-notebook is the classic way to end up with a figure nobody can reproduce.
</div>

`SENSOR_SET` selects all sensors or one of the official VectorView occipital, temporal or combined helmet selections. These are **sensor positions, not source-localized cortical ROIs**. `METRIC_PCA_MODE` controls whether participant-level metrics use the shared PCA, each participant's own PCA, or both.

In [ ]:
import os
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.ndimage import gaussian_filter1d

from coco_pipe.dim_reduction import DimReduction
from coco_pipe.viz.interactive import plot_scree, plot_trajectory
from pca_neural_trajectories.wakeman_henson import (
    LABEL_NAMES,
    MEG_SENSOR_SETS,
    preprocessing_config,
    _load_wakeman_henson_container,
    _preprocess_subject,
    epochs_path,
)

SEED = 42
RAW_ROOT = Path(os.getenv("MEG_RAW_ROOT", str(Path.home() / "mne_data" / "ds000117")))
POWER_DERIVATIVES = Path(
    os.getenv(
        "MEG_POWER_DERIVATIVES",
        str(RAW_ROOT / "derivatives" / "pca_trajectories_power"),
    )
)
# Every prepared participant; the loader silently skips ids that are absent,
# so this can stay at the full cohort on a machine holding only a subset.
SUBJECTS = tuple(f"{subject:02d}" for subject in range(1, 17))
SENSOR_SET = "all_sensors"

# The first run creates missing long epochs from local raw data.
# Existing derivatives are detected and skipped.
PREPARE_POWER_EPOCHS = os.getenv("MEG_PREPARE_POWER_EPOCHS", "1") == "1"

# Choose "shared", "subject", or "both".
METRIC_PCA_MODE = "both"

BANDS = {
    "alpha": (8.0, 12.0),
    "beta": (13.0, 30.0),
    "low_gamma": (30.0, 45.0),
}
N_COMPONENTS = 10
N_DISPLAY_COMPONENTS = 3
FINAL_WINDOW = (-0.2, 0.8)
BASELINE_WINDOW = (-0.2, 0.0)
ACTIVE_WINDOW = (0.0, 0.6)
ENVELOPE_SFREQ = 62.5
SMOOTHING_S = 0.04

CONDITION_COLORS = {1: "#0072B2", 2: "#D55E00", 3: "#009E73"}
CONDITION_FILLS = {
    1: "rgba(0, 114, 178, 0.15)",
    2: "rgba(213, 94, 0, 0.15)",
    3: "rgba(0, 158, 115, 0.15)",
}
CONTRAST_COLORS = {
    "Faces vs Scrambled": "#7B2CBF",
    "Famous vs Unfamiliar": "#D55E00",
    "Famous vs Scrambled": "#0072B2",
}

if METRIC_PCA_MODE not in {"shared", "subject", "both"}:
    raise ValueError('METRIC_PCA_MODE must be "shared", "subject", or "both".')
if SENSOR_SET not in MEG_SENSOR_SETS:
    raise ValueError(f"SENSOR_SET must be one of {tuple(MEG_SENSOR_SETS)}.")

pio.templates.default = "plotly_white"
rng = np.random.default_rng(SEED)

## Step 1. Choose the Representation — Recompute Long Epochs for Power

The final analysis window stays at `-0.2…0.8 s`, but filtering runs on `-1.2…1.8 s` epochs. The extra second on each side pushes the FIR and Hilbert boundary transients outside the plotted interval.

<div class="alert alert-danger">
<b>🚫 Why cropping first would corrupt the result:</b><br>
Filter edge effects are large, smooth and stimulus-time-locked by construction — they begin exactly where the epoch begins. Crop first and they land inside your window looking precisely like an evoked power change, in every condition, at the same latency. Padding is not a nicety here; it is what separates a response from an artifact of the filter.
</div>

The spectral preprocessing retains frequencies through 90 Hz, applies the same Maxwell/ICA cleaning as the broadband workflow, and deliberately applies **no** field baseline. Power is baselined only after the Hilbert transform.

<div class="alert alert-warning">
<b>⏱️ First-run cost:</b><br>
Preparing participants reruns substantial MNE preprocessing and can take well over an hour for a few participants. It uses the local raw data and downloads nothing. Existing derivatives are detected and reused, so a second run is fast.
</div>

In [ ]:
if PREPARE_POWER_EPOCHS:
    power_config = preprocessing_config(
        l_freq=0.5,
        h_freq=90.0,
        sfreq=250.0,
        tmin=-1.2,
        tmax=1.8,
        baseline=None,
        random_state=SEED,
    )
    for subject in SUBJECTS:
        _preprocess_subject(
            RAW_ROOT,
            subject,
            derivatives_root=POWER_DERIVATIVES,
            config=power_config,
            overwrite=False,
        )

missing = [
    subject
    for subject in SUBJECTS
    if not epochs_path(POWER_DERIVATIVES, subject).exists()
]
if missing:
    raise FileNotFoundError(
        "Missing long power epochs for "
        f"{missing}. Set PREPARE_POWER_EPOCHS=True and rerun this cell."
    )

print(f"Power derivatives: {POWER_DERIVATIVES}")
print(f"Selected subjects: {', '.join(SUBJECTS)}")

### Why one second of padding?

MNE derives the FIR filter length from the passband and transition bandwidth, so a narrow low-frequency band needs a longer filter than a wide high-frequency one — alpha is the binding constraint here.

Rather than assume every band is safe, we print the actual filter support. The half-support must sit comfortably inside the one-second padding on each side of the final crop; if it does not, the padding is too small for that band and the crop window is contaminated.

In [ ]:
filter_support = []
source_sfreq = 250.0
source_n_times = int(3.0 * source_sfreq) + 1

for band, (low, high) in BANDS.items():
    kernel = mne.filter.create_filter(
        np.empty((1, source_n_times)),
        sfreq=source_sfreq,
        l_freq=low,
        h_freq=high,
        fir_design="firwin",
        verbose=False,
    )
    filter_support.append({
        "band": band,
        "frequency_hz": f"{low:g}–{high:g}",
        "filter_length_s": len(kernel) / source_sfreq,
        "half_support_s": (len(kernel) - 1) / (2 * source_sfreq),
    })

display(pd.DataFrame(filter_support).round(3))

## Step 2. Preprocess & Normalize — Load the Whitened Trials

Each participant is whitened with that participant's empty-room covariance. Whitening is a spatial linear transform putting magnetometers and gradiometers on a common noise scale, so the later Hilbert modulus describes power in **noise-normalised sensor mixtures**, not physical sensor power. We do not apply a second channel z-score, which would replace measured noise scaling with an arbitrary one.

All retained trials are loaded with aligned channels and trial metadata, a 50-Hz notch suppresses line noise before the spectral transform, and the same trials are then transformed into alpha, beta and low gamma.

<div class="alert alert-warning">
<b>⚠️ The statistical unit is the participant:</b><br>
Condition trajectories are averaged within participant, and participants receive equal weight in every group summary. Time samples and trials are repeated, correlated observations — treating them as independent would inflate any test by orders of magnitude.
</div>

In [ ]:
container_power = _load_wakeman_henson_container(
    POWER_DERIVATIVES,
    subjects=SUBJECTS,
    notch_freq=50.0,
    sensor_set=SENSOR_SET,
)

source_X = np.asarray(container_power.X, dtype=np.float32)
source_times = np.asarray(container_power.coords["time"], dtype=float)
source_channels = np.asarray(container_power.coords["channel"]).astype(str)
source_subjects = np.asarray(container_power.coords["subject"]).astype(str)
source_labels = np.asarray(container_power.y, dtype=int)
source_repetitions = np.asarray(
    container_power.coords["repetition"]
).astype(str)

print(f"Loaded whitened tensor: {source_X.shape}")
print(f"Sensor set: {SENSOR_SET} ({len(source_channels)} sensors)")
print(f"Notch frequency: {container_power.meta['notch_freq_hz']:.0f} Hz")

In [ ]:
source_sfreq = 1 / np.diff(source_times).mean()
source_info = mne.create_info(
    source_channels.tolist(),
    sfreq=source_sfreq,
    ch_types="misc",
)
source_epochs = {}
for subject in SUBJECTS:
    rows = source_subjects == subject
    metadata = pd.DataFrame({
        "condition_id": source_labels[rows],
        "repetition": source_repetitions[rows],
    })
    source_epochs[subject] = mne.EpochsArray(
        source_X[rows],
        source_info,
        tmin=source_times[0],
        metadata=metadata,
        baseline=None,
        verbose=False,
    )

# Keep one participant available for the transformation demonstration.
demo_subject = SUBJECTS[0]
demo_source_epochs = source_epochs[demo_subject]

demo_labels = demo_source_epochs.metadata[
    "condition_id"
].to_numpy(int)
display(pd.Series(demo_labels).map(LABEL_NAMES).value_counts())

## Step 3. From Signed MEG to Band Power

For each band, MNE performs the band-pass and the Hilbert amplitude transform. Squaring the amplitude gives power. We then smooth power over 40 ms, express it in decibels relative to the pre-stimulus mean, crop only after those operations, and finally resample the slowly varying envelope:

$$P_{dB}(t)=10\log_{10}\left(\frac{P(t)}{\overline{P}_{-0.2:0}}\right).$$

The example below exposes every transformation for one trial and one sensor, which is worth reading carefully — the shape of the final curve is produced as much by these steps as by the brain. The dB value is a ratio to *that trial and sensor's own* baseline: 0 dB is baseline-level power, +3 dB is roughly twice the power, and negative values are decreases.

<div class="alert alert-secondary">
<b>💡 Why smooth before taking the ratio?</b><br>
The envelope is strictly positive but instantaneous power is extremely noisy, and a ratio with a noisy denominator is badly behaved. Smoothing first stabilizes the baseline-relative estimate. It also contributes directly to how smooth the resulting trajectories look, which is why smoothness in these figures must never be read as evidence of orderly neural dynamics.
</div>

<div class="alert alert-info">
<b>🧠 Why resample last?</b><br>
After band-passing and 40 ms smoothing, the envelope carries no information above a few tens of Hz, so keeping the original sampling rate stores mostly redundant samples. Resampling <i>after</i> the crop and the smoothing means the decimation never has to reach across the epoch boundary.
</div>

In [ ]:
demo_epochs = demo_source_epochs[0:1]
demo_times = demo_epochs.times
demo_signed = demo_epochs.get_data(copy=True)[0, 0]

demo_filtered_epochs = demo_epochs.copy().filter(
    *BANDS["alpha"],
    picks="all",
    verbose=False,
)
demo_filtered = demo_filtered_epochs.get_data(copy=True)[0, 0]
demo_envelope_epochs = demo_filtered_epochs.copy().apply_hilbert(
    picks="all",
    envelope=True,
    verbose=False,
)
demo_power = demo_envelope_epochs.get_data(copy=True)[0, 0] ** 2
demo_smoothed = gaussian_filter1d(
    demo_power,
    sigma=SMOOTHING_S * demo_epochs.info["sfreq"],
    mode="reflect",
)
demo_baseline = (
    (demo_times >= BASELINE_WINDOW[0])
    & (demo_times <= BASELINE_WINDOW[1])
)
demo_db = 10 * np.log10(
    np.maximum(demo_smoothed, np.finfo(float).tiny)
    / demo_smoothed[demo_baseline].mean()
)

fig_transform = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Whitened signed MEG",
        "Alpha-filtered signal",
        "Hilbert power",
        "Smoothed baseline-relative power",
    ),
)
for row, values, label in [
    (1, demo_signed, "signed"),
    (2, demo_filtered, "filtered"),
    (3, demo_power, "power"),
    (4, demo_db, "dB"),
]:
    fig_transform.add_trace(
        go.Scatter(
            x=demo_times,
            y=values,
            mode="lines",
            name=label,
            showlegend=False,
        ),
        row=row,
        col=1,
    )
fig_transform.add_vline(
    x=0, line_dash="dash", line_color="black"
)
fig_transform.update_xaxes(title_text="Time (s)", row=4, col=1)
fig_transform.update_layout(
    height=760,
    title="MNE alpha-power transformation: one trial, one sensor",
)
fig_transform.show()

## Step 4. Compute All Three Band-Power Tensors

The transform is applied one participant and one band at a time to bound memory: long epochs, three bands and 306 sensors do not all fit at once. MNE handles filtering, Hilbert envelopes, cropping and resampling; only the Gaussian smoothing and the explicit dB calculation operate on raw NumPy arrays.

Every band retains identical participants, trials, labels, repetitions, channels and final time coordinates. That correspondence is what makes the cross-band comparison in Step 11 a comparison of representations rather than of datasets.

In [ ]:
band_parts = {band: [] for band in BANDS}
count_rows = []

for subject_number, subject in enumerate(SUBJECTS):
    source = source_epochs[subject]
    source_labels = source.metadata["condition_id"].to_numpy(int)
    for condition in sorted(LABEL_NAMES):
        count_rows.append({
            "subject": subject,
            "condition": LABEL_NAMES[condition],
            "trials": int(np.sum(source_labels == condition)),
        })

    for band, (low, high) in BANDS.items():
        band_epochs = source.copy().filter(
            low,
            high,
            picks="all",
            verbose=False,
        )
        band_epochs.apply_hilbert(
            picks="all",
            envelope=True,
            verbose=False,
        )
        power = band_epochs.get_data(copy=True) ** 2
        power = gaussian_filter1d(
            power,
            sigma=SMOOTHING_S * source.info["sfreq"],
            axis=-1,
            mode="reflect",
        )
        source_times = source.times
        baseline_mask_source = (
            (source_times >= BASELINE_WINDOW[0])
            & (source_times <= BASELINE_WINDOW[1])
        )
        baseline_power = power[..., baseline_mask_source].mean(
            axis=-1,
            keepdims=True,
        )
        power_db = 10 * np.log10(
            np.maximum(power, np.finfo(float).tiny)
            / np.maximum(baseline_power, np.finfo(float).tiny)
        )

        power_epochs = mne.EpochsArray(
            power_db,
            source.info,
            events=source.events,
            event_id=source.event_id,
            tmin=source.tmin,
            metadata=source.metadata,
            baseline=None,
            verbose=False,
        )
        power_epochs.crop(*FINAL_WINDOW)
        power_epochs.resample(
            ENVELOPE_SFREQ,
            npad="auto",
            verbose=False,
        )

        band_parts[band].append({
            "X": power_epochs.get_data(copy=True).astype(np.float32),
            "times": power_epochs.times.copy(),
            "channels": np.asarray(power_epochs.ch_names),
            "labels": power_epochs.metadata[
                "condition_id"
            ].to_numpy(int),
            "repetitions": power_epochs.metadata[
                "repetition"
            ].astype(str).to_numpy(),
            "subjects": np.repeat(subject, len(power_epochs)),
        })

del demo_source_epochs, source_epochs, container_power, source_X

trial_counts = pd.DataFrame(count_rows).pivot(
    index="subject", columns="condition", values="trials"
)
display(trial_counts)

band_data = {}
for band, parts in band_parts.items():
    band_data[band] = {
        "X": np.concatenate([part["X"] for part in parts]),
        "times": parts[0]["times"],
        "channels": parts[0]["channels"],
        "labels": np.concatenate([part["labels"] for part in parts]),
        "repetitions": np.concatenate([
            part["repetitions"] for part in parts
        ]),
        "subjects": np.concatenate([
            part["subjects"] for part in parts
        ]),
    }
    print(band, band_data[band]["X"].shape)

### Sensor-space power check

Before any PCA, we average dB power across sensors within each participant and condition.

What to look for: a band-specific response emerging *after* onset. What would be alarming: a condition offset present throughout the baseline. Because every trial is baselined against its own pre-stimulus mean, a persistent pre-stimulus separation cannot be a response — it points instead at trial sampling, whitening, or a baseline window that overlaps the stimulus.

<div class="alert alert-info">
<b>🧪 QC, not inference:</b><br>
Averaging across sensors cancels spatially opposed effects, so a flat line here does not mean nothing happened. This panel checks scale, baseline behaviour and timing; it says nothing about whether the conditions are separable in the multivariate sensor pattern.
</div>

In [ ]:
fig_sensor_power = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=list(BANDS),
)

for row, (band, data) in enumerate(band_data.items(), start=1):
    for condition in sorted(LABEL_NAMES):
        curves = []
        for subject in SUBJECTS:
            rows = (
                (data["subjects"] == subject)
                & (data["labels"] == condition)
            )
            curves.append(data["X"][rows].mean(axis=(0, 1)))
        curves = np.asarray(curves)
        mean = curves.mean(axis=0)
        sem = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        color = CONDITION_COLORS[condition]
        fig_sensor_power.add_trace(
            go.Scatter(
                x=data["times"],
                y=mean,
                mode="lines",
                name=LABEL_NAMES[condition],
                legendgroup=LABEL_NAMES[condition],
                showlegend=row == 1,
                line=dict(color=color, width=2.5),
            ),
            row=row,
            col=1,
        )
        fig_sensor_power.add_trace(
            go.Scatter(
                x=data["times"],
                y=mean + sem,
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=1,
        )
        fig_sensor_power.add_trace(
            go.Scatter(
                x=data["times"],
                y=mean - sem,
                mode="lines",
                line=dict(width=0),
                fill="tonexty",
                fillcolor=CONDITION_FILLS[condition],
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=1,
        )

fig_sensor_power.add_vline(
    x=0, line_dash="dash", line_color="black"
)
fig_sensor_power.update_xaxes(title_text="Time (s)", row=3, col=1)
fig_sensor_power.update_yaxes(title_text="Mean power (dB)")
fig_sensor_power.update_layout(
    height=820,
    title="Whitened sensor-space band power",
)
fig_sensor_power.show()

## Step 4b. Fit One Shared PCA Per Band

Alpha, beta and low gamma have genuinely different covariance structures — different generators, different spatial topographies, different noise. Forcing them into one basis would let whichever band has the largest numerical variance dictate the axes for all three. Each band therefore receives its own PCA, shared across all three conditions and all selected participants.

<div class="alert alert-danger">
<b>🚫 Across bands, PC1 is not PC1:</b><br>
Each band's basis is fitted independently, so raw coordinates, distances and speeds are <b>not commensurate between bands</b>. Step 11 compares bands only through dimensionless descriptors.
</div>

We fit ten components to inspect the spectrum and display three. The participation ratio reported below is explicitly limited to that ten-component spectrum — it is a compact diagnostic, not an estimate of the full sensor-space intrinsic dimensionality.

In [ ]:
band_results = {}

for band, data in band_data.items():
    X = data["X"]
    n_trials, n_sensors, n_times = X.shape
    pooled = X.transpose(0, 2, 1).reshape(
        n_trials * n_times,
        n_sensors,
    )
    shared_pca = DimReduction(
        method="PCA",
        n_components=N_COMPONENTS,
        random_state=SEED,
    )
    scores_flat = shared_pca.fit_transform(pooled)
    scores = scores_flat.reshape(
        n_trials,
        n_times,
        N_COMPONENTS,
    )
    baseline = data["times"] < 0
    scores -= scores[:, baseline].mean(axis=1, keepdims=True)
    evr = np.asarray(
        shared_pca.get_diagnostics()["explained_variance_ratio_"]
    )

    band_results[band] = {
        **data,
        "shared_pca": shared_pca,
        "shared_scores": scores,
        "evr": evr,
    }

    figure = plot_scree(evr)
    figure.update_layout(title=f"{band}: shared PCA spectrum")
    figure.show()

pca_diagnostics = pd.DataFrame([
    {
        "band": band,
        "variance_3pc": result["evr"][:3].sum(),
        "variance_10pc": result["evr"].sum(),
        "participation_ratio_10pc": (
            result["evr"].sum() ** 2
            / np.square(result["evr"]).sum()
        ),
    }
    for band, result in band_results.items()
])
display(pca_diagnostics.round(3))

## Step 5. Select Components — Fit Participant-Specific PCA Models

When enabled, each participant receives one PCA per band, fitted across all three of their conditions. These axes are used **only** for rotation-invariant within-participant distances and speeds; participant-specific PC coordinates are never averaged into a group trajectory, because one participant's PC1 is not another's.

<div class="alert alert-secondary">
<b>🔍 What this sensitivity actually tests:</b><br>
Euclidean distance and speed ignore PC sign and any rotation <i>within</i> the retained subspace, so they are safe to compute in each participant's own basis. What they still depend on is <b>which</b> three-dimensional subspace was retained. Agreement between the shared-PCA and participant-PCA metrics therefore shows that a conclusion is not an artifact of the pooled subspace choice.
</div>

In [ ]:
for band, result in band_results.items():
    result["subject_scores"] = None
    result["subject_pcas"] = {}

    if METRIC_PCA_MODE in {"subject", "both"}:
        subject_scores = np.empty_like(result["shared_scores"])
        for subject in SUBJECTS:
            rows = result["subjects"] == subject
            X_subject = result["X"][rows]
            n_subject_trials = X_subject.shape[0]
            subject_matrix = X_subject.transpose(0, 2, 1).reshape(
                n_subject_trials * X_subject.shape[2],
                X_subject.shape[1],
            )
            subject_pca = DimReduction(
                method="PCA",
                n_components=N_COMPONENTS,
                random_state=SEED,
            )
            subject_flat = subject_pca.fit_transform(subject_matrix)
            subject_traj = subject_flat.reshape(
                n_subject_trials,
                X_subject.shape[2],
                N_COMPONENTS,
            )
            baseline = result["times"] < 0
            subject_traj -= subject_traj[:, baseline].mean(
                axis=1,
                keepdims=True,
            )
            subject_scores[rows] = subject_traj
            result["subject_pcas"][subject] = subject_pca
        result["subject_scores"] = subject_scores

subject_variance_rows = []
for band, result in band_results.items():
    for subject, subject_pca in result["subject_pcas"].items():
        evr = np.asarray(
            subject_pca.get_diagnostics()["explained_variance_ratio_"]
        )
        subject_variance_rows.append({
            "band": band,
            "subject": subject,
            "variance_3pc": evr[:3].sum(),
        })

if subject_variance_rows:
    display(pd.DataFrame(subject_variance_rows).round(3))
else:
    print("Participant-specific PCA skipped.")

## Step 6. Project and Inspect — PC Timecourses and Sensor Loadings

Before interpreting any trajectory, look at what the axes are. The shared PC traces show *when* each low-dimensional axis changes, and the loading tables identify strongly weighted sensors.

Signs are arbitrary, so read relative timing and geometry rather than positive-versus-negative polarity. And a large loading marks a contribution to a sensor-space variance pattern — it does not give that PC biological ownership of a cortical region, still less localise a generator.

In [ ]:
for band, result in band_results.items():
    figure = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("PC1", "PC2", "PC3"),
    )
    for pc in range(3):
        for condition in sorted(LABEL_NAMES):
            curves = []
            for subject in SUBJECTS:
                rows = (
                    (result["subjects"] == subject)
                    & (result["labels"] == condition)
                )
                curves.append(
                    result["shared_scores"][rows].mean(axis=0)[:, pc]
                )
            curves = np.asarray(curves)
            mean = curves.mean(axis=0)
            figure.add_trace(
                go.Scatter(
                    x=result["times"],
                    y=mean,
                    mode="lines",
                    name=LABEL_NAMES[condition],
                    legendgroup=LABEL_NAMES[condition],
                    showlegend=pc == 0,
                    line=dict(
                        color=CONDITION_COLORS[condition],
                        width=2.5,
                    ),
                ),
                row=pc + 1,
                col=1,
            )
    figure.add_vline(
        x=0, line_dash="dash", line_color="black"
    )
    figure.update_xaxes(title_text="Time (s)", row=3, col=1)
    figure.update_yaxes(title_text="PC score")
    figure.update_layout(
        height=700,
        title=f"{band}: shared-PC timecourses",
    )
    figure.show()

loading_rows = []
for band, result in band_results.items():
    loadings = np.asarray(
        result["shared_pca"].get_components()
    )[:3]
    for pc, weights in enumerate(loadings, start=1):
        strongest = np.argsort(np.abs(weights))[-5:][::-1]
        loading_rows.extend({
            "band": band,
            "component": f"PC{pc}",
            "sensor": result["channels"][index],
            "loading": weights[index],
        } for index in strongest)

display(pd.DataFrame(loading_rows).round({"loading": 3}))

## Step 7. Plot Group-Mean Trajectories in Each Shared Band Space

Trials are averaged within participant and condition, then participants are averaged with equal weight. Alpha, beta and low gamma get separate panels because their PC axes are unrelated.

<div class="alert alert-info">
<b>💡 How to read these plots:</b><br>
<ul style="margin-bottom: 0;">
<li><b>Follow time within a panel.</b> Ask when the trajectory leaves baseline, whether conditions branch, and whether they return along a similar path.</li>
<li><b>Do not compare across panels</b> in absolute position, rotation or path length — only in qualitative topology and timing.</li>
<li><b>Smooth loops are partly manufactured.</b> Envelope estimation and 40 ms smoothing impose temporal continuity by construction. A smooth curve here is weaker evidence of orderly dynamics than the same curve would be in the broadband analysis.</li>
</ul>
</div>

In [ ]:
for band, result in band_results.items():
    group = []
    sem = []
    for condition in sorted(LABEL_NAMES):
        subject_means = np.stack([
            result["shared_scores"][
                (result["subjects"] == subject)
                & (result["labels"] == condition)
            ].mean(axis=0)
            for subject in SUBJECTS
        ])
        group.append(subject_means.mean(axis=0))
        sem.append(
            subject_means.std(axis=0, ddof=1)
            / np.sqrt(len(SUBJECTS))
        )
    result["group_trajectories"] = np.asarray(group)
    result["group_sem"] = np.asarray(sem)

    labels_plot = np.array([
        LABEL_NAMES[condition] for condition in sorted(LABEL_NAMES)
    ])
    colors_plot = {
        LABEL_NAMES[condition]: CONDITION_COLORS[condition]
        for condition in sorted(LABEL_NAMES)
    }
    figure = plot_trajectory(
        X=result["group_trajectories"][..., :2],
        times=result["times"],
        labels=labels_plot,
        sem=result["group_sem"][..., :2],
        color_map=colors_plot,
        title=f"{band}: equal-participant PC1–PC2 trajectories",
        dimensions=2,
        show_markers=True,
        add_start_end_markers=True,
    )
    figure.show()

In [ ]:
for band, result in band_results.items():
    labels_plot = np.array([
        LABEL_NAMES[condition] for condition in sorted(LABEL_NAMES)
    ])
    colors_plot = {
        LABEL_NAMES[condition]: CONDITION_COLORS[condition]
        for condition in sorted(LABEL_NAMES)
    }
    figure = plot_trajectory(
        X=result["group_trajectories"][..., :3],
        times=result["times"],
        labels=labels_plot,
        color_map=colors_plot,
        title=f"{band}: equal-participant PC1–PC2–PC3 trajectories",
        dimensions=3,
        show_markers=True,
        add_start_end_markers=True,
        height=650,
    )
    figure.show()

## Step 8. Compare Conditions — Planned Contrasts in Shared and Subject PCA

Within each band and each PCA metric space we compute two predeclared participant-level contrasts:

- **Famous vs Unfamiliar** compares the two face categories directly.
- **Faces vs Scrambled** first averages Famous and Unfamiliar with equal weight, then measures distance to Scrambled.

Distances are expressed relative to each participant's pre-stimulus mean, so the curves start at zero and any lift is a response rather than an offset.

Thin lines show individual participants and the thick line with its SEM summarizes them. Read both: a group mean built from one large individual effect and several flat ones tells a very different story from the same mean built from consistent small effects, and only the thin lines distinguish those cases.

In [ ]:
def contrast_curves(trajectories, result):
    curves = {
        "Faces vs Scrambled": [],
        "Famous vs Unfamiliar": [],
    }
    baseline = result["times"] < 0
    for subject in SUBJECTS:
        means = {
            condition: trajectories[
                (result["subjects"] == subject)
                & (result["labels"] == condition)
            ].mean(axis=0)
            for condition in sorted(LABEL_NAMES)
        }
        faces = 0.5 * (means[1] + means[2])
        distances = {
            "Faces vs Scrambled": np.linalg.norm(
                faces - means[3], axis=1
            ),
            "Famous vs Unfamiliar": np.linalg.norm(
                means[1] - means[2], axis=1
            ),
        }
        for name, distance in distances.items():
            curves[name].append(
                distance - distance[baseline].mean()
            )
    return {
        name: np.asarray(values)
        for name, values in curves.items()
    }


for band, result in band_results.items():
    spaces = {}
    if METRIC_PCA_MODE in {"shared", "both"}:
        spaces["Shared PCA"] = result["shared_scores"][..., :3]
    if METRIC_PCA_MODE in {"subject", "both"}:
        spaces["Subject PCA"] = result["subject_scores"][..., :3]
    result["metric_spaces"] = spaces
    result["contrast_curves"] = {
        space: contrast_curves(trajectories, result)
        for space, trajectories in spaces.items()
    }

In [ ]:
contrast_names = ["Faces vs Scrambled", "Famous vs Unfamiliar"]
for band, result in band_results.items():
    figure = make_subplots(
        rows=len(result["contrast_curves"]),
        cols=2,
        shared_xaxes=True,
        shared_yaxes=True,
        subplot_titles=[
            f"{space}: {contrast}"
            for space in result["contrast_curves"]
            for contrast in contrast_names
        ],
    )
    for row, (space, space_curves) in enumerate(
        result["contrast_curves"].items(), start=1
    ):
        for column, contrast in enumerate(
            contrast_names, start=1
        ):
            curves = space_curves[contrast]
            color = CONTRAST_COLORS[contrast]
            for subject, curve in zip(SUBJECTS, curves):
                figure.add_trace(
                    go.Scatter(
                        x=result["times"],
                        y=curve,
                        mode="lines",
                        line=dict(color=color, width=1),
                        opacity=0.28,
                        showlegend=False,
                        name=f"sub-{subject}",
                    ),
                    row=row,
                    col=column,
                )
            mean = curves.mean(axis=0)
            sem = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
            figure.add_trace(
                go.Scatter(
                    x=result["times"],
                    y=mean,
                    mode="lines",
                    line=dict(color=color, width=4),
                    showlegend=False,
                    name=contrast,
                ),
                row=row,
                col=column,
            )
            figure.add_trace(
                go.Scatter(
                    x=result["times"],
                    y=mean + sem,
                    mode="lines",
                    line=dict(width=0),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=row,
                col=column,
            )
            figure.add_trace(
                go.Scatter(
                    x=result["times"],
                    y=mean - sem,
                    mode="lines",
                    line=dict(width=0),
                    fill="tonexty",
                    fillcolor="rgba(123, 44, 191, 0.12)",
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=row,
                col=column,
            )
    figure.add_vline(
        x=0, line_dash="dash", line_color="black"
    )
    figure.add_hline(
        y=0, line_dash="dot", line_color="grey"
    )
    figure.update_xaxes(title_text="Time (s)")
    figure.update_yaxes(
        title_text="Distance relative to baseline", col=1
    )
    figure.update_layout(
        height=380 * len(result["contrast_curves"]),
        title=f"{band}: planned contrasts",
    )
    figure.show()

In [ ]:
metric_rows = []
for band, result in band_results.items():
    times = result["times"]
    active = (
        (times >= ACTIVE_WINDOW[0])
        & (times <= ACTIVE_WINDOW[1])
    )
    active_times = times[active]
    for space, space_curves in result["contrast_curves"].items():
        for contrast, curves in space_curves.items():
            for subject, curve in zip(SUBJECTS, curves):
                active_curve = curve[active]
                peak = int(np.argmax(active_curve))
                metric_rows.append({
                    "band": band,
                    "pca_space": space,
                    "subject": subject,
                    "contrast": contrast,
                    "auc_0_600ms": np.trapezoid(
                        active_curve, active_times
                    ),
                    "peak_separation": active_curve[peak],
                    "peak_time_s": active_times[peak],
                })

metric_summary = pd.DataFrame(metric_rows)
display(metric_summary.round(3))
display(
    metric_summary.groupby(
        ["band", "pca_space", "contrast"]
    )
    .agg(
        mean_auc=("auc_0_600ms", "mean"),
        sem_auc=("auc_0_600ms", "sem"),
        mean_peak_time_s=("peak_time_s", "mean"),
    )
    .round(3)
)

### Trajectory speed is descriptive

Speed is the norm of the numerical derivative of each participant's condition-mean trajectory.

<div class="alert alert-success">
<b>🧠 A shared speed profile is a finding, not a failure:</b><br>
The largest dynamical component of a population response is often <b>condition-invariant</b> — a stereotyped state transition occurring on every trial regardless of stimulus, occupying dimensions orthogonal to the tuned ones. Speed curves that share a profile across conditions are the expected signature of such a component: the conditions differ in <i>where</i> the trajectory goes, not in <i>how fast</i>.
</div>

Numerical differentiation amplifies noise, and each band/PCA space carries its own coordinate scale. Interpret broad within-panel timing, never isolated peaks or raw cross-band magnitudes.

In [ ]:
for band, result in band_results.items():
    figure = make_subplots(
        rows=len(result["metric_spaces"]),
        cols=1,
        shared_xaxes=True,
        subplot_titles=list(result["metric_spaces"]),
    )
    for row, (space, trajectories) in enumerate(
        result["metric_spaces"].items(), start=1
    ):
        for condition in sorted(LABEL_NAMES):
            curves = []
            for subject in SUBJECTS:
                subject_mean = trajectories[
                    (result["subjects"] == subject)
                    & (result["labels"] == condition)
                ].mean(axis=0)
                velocity = np.gradient(
                    subject_mean,
                    result["times"],
                    axis=0,
                )
                curves.append(np.linalg.norm(velocity, axis=1))
            curves = np.asarray(curves)
            figure.add_trace(
                go.Scatter(
                    x=result["times"],
                    y=curves.mean(axis=0),
                    mode="lines",
                    name=LABEL_NAMES[condition],
                    legendgroup=LABEL_NAMES[condition],
                    showlegend=row == 1,
                    line=dict(
                        color=CONDITION_COLORS[condition],
                        width=2.5,
                    ),
                ),
                row=row,
                col=1,
            )
    figure.add_vline(
        x=0, line_dash="dash", line_color="black"
    )
    figure.update_xaxes(title_text="Time (s)")
    figure.update_yaxes(title_text="Speed (a.u./s)")
    figure.update_layout(
        height=360 * len(result["metric_spaces"]),
        title=f"{band}: trajectory speed",
    )
    figure.show()

## Step 9. Fit Focused Two-Condition PCA Spaces

Within every band we additionally fit two shared PCAs across the selected participants:

1. **Famous and Unfamiliar only**
2. **Famous and Scrambled only**

A three-condition fit allocates its axes to whatever varies most overall, which need not be what distinguishes a particular pair. A focused fit can surface pair-relevant variance that was lower-ranked in the pooled space.

<div class="alert alert-warning">
<b>⚠️ Each focused model is its own coordinate system:</b><br>
Coordinates and raw distances cannot be compared with another PCA model, including the three-condition one. Compare <b>divergence timing and consistency</b>. Agreement with the primary space strengthens the descriptive result; an effect visible only after focused refitting is markedly more model-dependent and should be reported as such.
</div>

In [ ]:
FOCUSED_PAIRS = {
    "Famous vs Unfamiliar": (1, 2),
    "Famous vs Scrambled": (1, 3),
}
focused_results = {}

for band, data in band_data.items():
    for pair_name, pair in FOCUSED_PAIRS.items():
        keep = np.isin(data["labels"], pair)
        X_pair = data["X"][keep]
        labels_pair = data["labels"][keep]
        subjects_pair = data["subjects"][keep]
        pair_matrix = X_pair.transpose(0, 2, 1).reshape(
            X_pair.shape[0] * X_pair.shape[2],
            X_pair.shape[1],
        )
        pair_pca = DimReduction(
            method="PCA",
            n_components=N_COMPONENTS,
            random_state=SEED,
        )
        pair_flat = pair_pca.fit_transform(pair_matrix)
        pair_scores = pair_flat.reshape(
            X_pair.shape[0],
            X_pair.shape[2],
            N_COMPONENTS,
        )
        baseline = data["times"] < 0
        pair_scores -= pair_scores[:, baseline].mean(
            axis=1,
            keepdims=True,
        )

        subject_means = {
            (subject, condition): pair_scores[
                (subjects_pair == subject)
                & (labels_pair == condition)
            ].mean(axis=0)
            for subject in SUBJECTS
            for condition in pair
        }
        group = np.stack([
            np.stack([
                subject_means[subject, condition]
                for subject in SUBJECTS
            ]).mean(axis=0)
            for condition in pair
        ])
        sem = np.stack([
            np.stack([
                subject_means[subject, condition]
                for subject in SUBJECTS
            ]).std(axis=0, ddof=1) / np.sqrt(len(SUBJECTS))
            for condition in pair
        ])
        separation = np.stack([
            np.linalg.norm(
                subject_means[subject, pair[0]][:, :3]
                - subject_means[subject, pair[1]][:, :3],
                axis=1,
            )
            for subject in SUBJECTS
        ])
        separation -= separation[:, baseline].mean(
            axis=1,
            keepdims=True,
        )

        focused_results[band, pair_name] = {
            "pair": pair,
            "pca": pair_pca,
            "group": group,
            "sem": sem,
            "separation": separation,
            "times": data["times"],
        }

In [ ]:
focused_rows = []
for (band, pair_name), result in focused_results.items():
    pair = result["pair"]
    labels_plot = np.array([LABEL_NAMES[c] for c in pair])
    colors_plot = {
        LABEL_NAMES[c]: CONDITION_COLORS[c] for c in pair
    }
    figure = plot_trajectory(
        X=result["group"][..., :2],
        times=result["times"],
        labels=labels_plot,
        sem=result["sem"][..., :2],
        color_map=colors_plot,
        title=f"{band} focused PCA: {pair_name}",
        dimensions=2,
        show_markers=True,
        add_start_end_markers=True,
    )
    figure.show()

    active = (
        (result["times"] >= ACTIVE_WINDOW[0])
        & (result["times"] <= ACTIVE_WINDOW[1])
    )
    for subject, curve in zip(SUBJECTS, result["separation"]):
        active_curve = curve[active]
        peak = int(np.argmax(active_curve))
        focused_rows.append({
            "band": band,
            "focused_space": pair_name,
            "subject": subject,
            "auc_0_600ms": np.trapezoid(
                active_curve,
                result["times"][active],
            ),
            "peak_time_s": result["times"][active][peak],
        })

focused_summary = pd.DataFrame(focused_rows)
display(
    focused_summary.groupby(["band", "focused_space"])
    .agg(
        mean_auc=("auc_0_600ms", "mean"),
        sem_auc=("auc_0_600ms", "sem"),
        mean_peak_time_s=("peak_time_s", "mean"),
    )
    .round(3)
)

## Step 10. Validate — Compare Bands Without Equating Their Axes

Raw PC coordinates, distance units and speeds are specific to each band, so a cross-band comparison has to run on dimensionless quantities: explained-variance structure, peak timing, separation-timecourse shape, and a normalised roughness descriptor.

Roughness divides accumulated second-difference magnitude by path length. That removes the scale but not the sampling rate or the smoothing kernel, both of which were chosen identically across bands precisely so that this descriptor stays comparable.

<div class="alert alert-warning">
<b>⚠️ Low gamma remains exploratory:</b><br>
Even with recomputed 90-Hz epochs and a 50-Hz notch, high-frequency sensor power is far more vulnerable to residual muscle activity and line-noise harmonics than alpha or beta. Muscle artifact is broadband and strongest exactly where low gamma lives. Treat a low-gamma-only effect with more suspicion than the same effect in alpha.
</div>

The cross-band tables organize descriptors; they do not place the three PCA spaces on a common ruler.

In [ ]:
cross_band_rows = []
for band, result in band_results.items():
    for contrast, curves in result["contrast_curves"][
        next(iter(result["contrast_curves"]))
    ].items():
        mean_curve = curves.mean(axis=0)
        active = (
            (result["times"] >= ACTIVE_WINDOW[0])
            & (result["times"] <= ACTIVE_WINDOW[1])
        )
        peak = int(np.argmax(mean_curve[active]))
        cross_band_rows.append({
            "band": band,
            "contrast": contrast,
            "peak_time_s": result["times"][active][peak],
            "variance_3pc": result["evr"][:3].sum(),
            "participation_ratio_10pc": (
                result["evr"].sum() ** 2
                / np.square(result["evr"]).sum()
            ),
        })

    for condition_index, condition in enumerate(sorted(LABEL_NAMES)):
        trajectory = result["group_trajectories"][
            condition_index, :, :3
        ]
        first_difference = np.diff(trajectory, axis=0)
        second_difference = np.diff(trajectory, n=2, axis=0)
        roughness = (
            np.linalg.norm(second_difference, axis=1).sum()
            / np.maximum(
                np.linalg.norm(first_difference, axis=1).sum(),
                np.finfo(float).eps,
            )
        )
        cross_band_rows.append({
            "band": band,
            "contrast": f"{LABEL_NAMES[condition]} trajectory",
            "normalised_roughness": roughness,
        })

cross_band_summary = pd.DataFrame(cross_band_rows)
display(cross_band_summary.round(3))

## Step 11. Validate — Family-Corrected Permutation Test

The formal test reuses the already-fitted shared PCA in each band, which is legitimate because the PCA never saw a label and therefore cannot change under a label shuffle.

Labels are shuffled **within participant and repetition**, preserving each participant's trial counts and preprocessing history, and the **maximum AUC across all three bands and both planned contrasts** forms the null.

<div class="alert alert-success">
<b>✅ Why the maximum, across the whole family:</b><br>
Six scalar tests are being run (3 bands × 2 contrasts). Testing each against its own null would let the best of six pass by chance roughly a quarter of the time at α = 0.05. Taking the maximum across the family into the null controls all six simultaneously — the same max-statistic logic used for the time axis elsewhere in the series.
</div>

<div class="alert alert-warning">
<b>⚠️ The p-value floor:</b><br>
Two hundred permutations are tutorial-scale and limit the smallest attainable corrected p-value to 1/201 ≈ 0.005. Report a result that beats every permutation as <b>p &lt; 0.005</b>, not as the literal printed value. The result is also conditional on these participants: the permutation controls the label mapping, not the sample.
</div>

In [ ]:
RUN_PERMUTATIONS = False
N_PERMUTATIONS = 200
spectral_inference = pd.DataFrame()

if RUN_PERMUTATIONS:
    observed = {}
    for band, result in band_results.items():
        active = (
            (result["times"] >= ACTIVE_WINDOW[0])
            & (result["times"] <= ACTIVE_WINDOW[1])
        )
        for contrast, curves in contrast_curves(
            result["shared_scores"][..., :3],
            result,
        ).items():
            observed[band, contrast] = np.trapezoid(
                curves.mean(axis=0)[active],
                result["times"][active],
            )

    family_null = []
    for _ in range(N_PERMUTATIONS):
        null_values = []
        for band, result in band_results.items():
            shuffled = result["labels"].copy()
            for subject in SUBJECTS:
                for repetition in np.unique(
                    result["repetitions"][
                        result["subjects"] == subject
                    ]
                ):
                    rows = np.flatnonzero(
                        (result["subjects"] == subject)
                        & (result["repetitions"] == repetition)
                    )
                    shuffled[rows] = rng.permutation(shuffled[rows])

            shuffled_result = {**result, "labels": shuffled}
            active = (
                (result["times"] >= ACTIVE_WINDOW[0])
                & (result["times"] <= ACTIVE_WINDOW[1])
            )
            curves = contrast_curves(
                result["shared_scores"][..., :3],
                shuffled_result,
            )
            null_values.extend(
                np.trapezoid(
                    values.mean(axis=0)[active],
                    result["times"][active],
                )
                for values in curves.values()
            )
        family_null.append(max(null_values))

    family_null = np.asarray(family_null)
    spectral_inference = pd.DataFrame([
        {
            "band": band,
            "contrast": contrast,
            "observed_auc": value,
            "p_family_corrected": (
                1 + np.sum(family_null >= value)
            ) / (N_PERMUTATIONS + 1),
        }
        for (band, contrast), value in observed.items()
    ])
    display(spectral_inference.round(4))
else:
    print("Permutation test skipped.")

## Step 12. Export the Analysis

The optional notebook export is deliberately compact: processed power tensors, shared and participant scores, PCA loadings, and the participant-level summary tables — not raw data or preprocessing intermediates.

For a reproducible end-to-end run, use the headless companion described at the end of this notebook. It mirrors every step and saves all figures, time-resolved tables, reducers, null distributions, manifests and a self-contained HTML report.

In [ ]:
SAVE_RESULTS = False

if SAVE_RESULTS:
    output_dir = Path(
        "outputs/tutorial_megfaces_spectral_envelopes"
    ) / SENSOR_SET
    output_dir.mkdir(parents=True, exist_ok=True)

    for band, result in band_results.items():
        np.savez_compressed(
            output_dir / f"{band}_pca_power.npz",
            power_db=result["X"].astype(np.float32),
            times=result["times"],
            labels=result["labels"],
            subjects=result["subjects"],
            sensor_set=SENSOR_SET,
            shared_scores=result["shared_scores"].astype(np.float32),
            subject_scores=(
                result["subject_scores"].astype(np.float32)
                if result["subject_scores"] is not None
                else np.array([])
            ),
            shared_loadings=np.asarray(
                result["shared_pca"].get_components()
            ),
            explained_variance_ratio=result["evr"],
        )

    metric_summary.to_csv(
        output_dir / "planned_contrasts.csv",
        index=False,
    )
    focused_summary.to_csv(
        output_dir / "focused_pca_contrasts.csv",
        index=False,
    )
    if not spectral_inference.empty:
        spectral_inference.to_csv(
            output_dir / "family_corrected_inference.csv",
            index=False,
        )
    print(f"Saved compact results to {output_dir}")
else:
    print("SAVE_RESULTS is False; nothing was written.")

## Conclusions & Interpretation Checklist

The tutorial has separated three questions that are easy to conflate:

- **Which band-power trajectories are visible in one shared three-condition space?**
- **Do participant-level metrics depend on a shared or participant-specific PCA subspace?**
- **Does a focused two-condition PCA reveal structure that is lower-ranked in the three-condition fit?**

Before interpreting a spectral trajectory, check:

- **Padding.** The filter's half-support must sit inside the padding, or the crop window contains filter transients rather than responses (Step 1).
- **Baseline behaviour.** A condition difference present before onset is not a response; it is a sampling or preprocessing problem (Step 4's QC panel).
- **Smoothness is manufactured.** Amplitude envelopes plus 40 ms smoothing impose temporal continuity. Smooth trajectories here are not evidence of orderly dynamics.
- **Bands are separate coordinate systems.** Compare timing, shape and dimensionless descriptors, never raw distances (Step 10).
- **Multiplicity was handled as one family.** Six scalar tests share one max-statistic null (Step 11), and the reported p-value has a floor.
- **Low gamma is the least trustworthy band** and should carry the most caveats.
- **Sensor-space band power does not localise sources.**

<div class="alert alert-success">
<b>🎯 Main takeaway:</b><br>
The same 10-step workflow transfers intact from evoked fields to induced band power — what changes is the meaning of a coordinate, and with it the set of artifacts that can imitate a result. State the representation before stating the finding.
</div>

## Running the Same Analysis Headlessly

The companion script repeats every computation in this notebook and writes the full output bundle plus a self-contained HTML report:

```bash
python scripts/analysis_megfaces_spectral_envelopes.py \
    --output outputs/megfaces_spectral_envelopes
```

<div class="alert alert-warning">
<b>💾 Resource note:</b><br>
Long epochs plus a per-band Hilbert envelope over 306 sensors are memory-hungry — considerably more so than the broadband analysis. On a cluster, request a high-memory partition rather than a standard one; this step is the most likely in the whole series to be killed for exceeding its allocation.
</div>